In [1]:
%%writefile matmul.cu
#include <iostream>
#include <vector>
#include <chrono>
#include <cuda_runtime.h>
#include <iomanip>

using namespace std;

#define TILE 16

// ================= CPU =================
void matmul_cpu(const vector<float>& A, const vector<float>& B, vector<float>& C, int N) {
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++) {
            float sum = 0;
            for (int k = 0; k < N; k++)
                sum += A[i * N + k] * B[k * N + j];
            C[i * N + j] = sum;
        }
}

float sum_matrix(const vector<float>& M) {
    float s = 0;
    for (float x : M) s += x;
    return s;
}

// ================= GPU =================
__global__ void matmul_gpu(float* A, float* B, float* C, int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;

    float sum = 0;

    for (int t = 0; t < (N + TILE - 1) / TILE; t++) {

        if (row < N && t * TILE + threadIdx.x < N)
            As[threadIdx.y][threadIdx.x] = A[row * N + t * TILE + threadIdx.x];
        else
            As[threadIdx.y][threadIdx.x] = 0;

        if (col < N && t * TILE + threadIdx.y < N)
            Bs[threadIdx.y][threadIdx.x] = B[(t * TILE + threadIdx.y) * N + col];
        else
            Bs[threadIdx.y][threadIdx.x] = 0;

        __syncthreads();

        for (int k = 0; k < TILE; k++)
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];

        __syncthreads();
    }

    if (row < N && col < N)
        C[row * N + col] = sum;
}

int main() {

    cout << setw(8) << "N"
         << setw(15) << "CPU(s)"
         << setw(15) << "GPU(ms)"
         << setw(15) << "Speedup"
         << setw(20) << "CPU sum"
         << setw(20) << "GPU sum"
         << endl;

    for (int N = 100; N <= 2000; N += 100) {

        cout << "\n=== Iteration N = " << N << " ===" << endl;

        size_t size = N * N * sizeof(float);

        vector<float> A(N*N), B(N*N), C_cpu(N*N), C_gpu(N*N);

        for (int i = 0; i < N*N; i++) {
            A[i] = rand() % 5;
            B[i] = rand() % 5;
        }

        // ===== CPU =====
        auto cpu_start = chrono::high_resolution_clock::now();
        matmul_cpu(A, B, C_cpu, N);
        auto cpu_end = chrono::high_resolution_clock::now();

        double cpu_time = chrono::duration<double>(cpu_end - cpu_start).count();
        float cpu_sum = sum_matrix(C_cpu);

        // ===== GPU =====
        float *d_A, *d_B, *d_C;
        cudaMalloc(&d_A, size);
        cudaMalloc(&d_B, size);
        cudaMalloc(&d_C, size);

        cudaMemcpy(d_A, A.data(), size, cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, B.data(), size, cudaMemcpyHostToDevice);

        dim3 threads(TILE, TILE);
        dim3 blocks((N + TILE - 1) / TILE, (N + TILE - 1) / TILE);

        cudaEvent_t start, stop;
        cudaEventCreate(&start);
        cudaEventCreate(&stop);

        cudaEventRecord(start);

        matmul_gpu<<<blocks, threads>>>(d_A, d_B, d_C, N);

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float gpu_time;
        cudaEventElapsedTime(&gpu_time, start, stop);

        cudaMemcpy(C_gpu.data(), d_C, size, cudaMemcpyDeviceToHost);
        float gpu_sum = sum_matrix(C_gpu);

        double speedup = cpu_time / (gpu_time / 1000.0);

        cout << setw(8) << N
             << setw(15) << cpu_time
             << setw(15) << gpu_time
             << setw(15) << speedup
             << setw(20) << cpu_sum
             << setw(20) << gpu_sum
             << endl;

        cudaFree(d_A);
        cudaFree(d_B);
        cudaFree(d_C);
    }

    return 0;
}

Overwriting matmul.cu


In [3]:
!nvcc matmul.cu -o matmul
!./matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
       N         CPU(s)        GPU(ms)        Speedup             CPU sum             GPU sum

=== Iteration N = 100 ===
     100     0.00681793       0.167968        40.5906         4.01806e+06         4.01806e+06

=== Iteration N = 200 ===
     200      0.0564879       0.050944        1108.82         3.18667e+07         3.18667e+07

=== Iteration N = 300 ===
     300       0.180876        0.10944        1652.74         1.08271e+08         1.08271e+08

=== Iteration N = 400 ===
     400       0.446745        0.20944        2133.04          2.5654e+08          2.5654e+08

=== Iteration N = 500 ===
     500       0.954839       0.442944        2155.66         5.01704e+08         5.01704e+08

=== Iteration N = 600 ===
     600          1.671         1.2543        1332.22         8.63572e+08         8.635